In [1]:
import os
import json
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from torch.utils.data import Dataset, DataLoader, random_split

In [2]:
# STEP 0 — mount (ทำครั้งเดียวต่อ session)
from google.colab import drive
drive.mount('/content/drive')

# STEP 1 — ตรวจว่าพาธถูกจริงไหม
!ls "/content/drive/MyDrive/Project Deeplearning/TACO" | head

# STEP 2 — ถ้ารายการโผล่ถูก ให้รันโค้ดต่อได้เลย
from pathlib import Path, PurePosixPath
ANN_FILE = Path("/content/drive/MyDrive/Project Deeplearning/TACO/train_annotations.json")
print("exists?", ANN_FILE.exists())        # ต้องขึ้น True

with open(ANN_FILE, "r") as f:
    taco_json = json.load(f)
category_map = {c["id"]: c["name"] for c in taco_json["categories"]}
print("✓ loaded", len(category_map), "classes")


Mounted at /content/drive
annotations.json
batch_1
batch_10
batch_11
batch_12
batch_13
batch_14
batch_15
batch_2
batch_3
exists? True
✓ loaded 60 classes


In [4]:
# สมมติเราเซ็ต ROOT ไว้แล้วในเซลล์ถัดไปว่า
ROOT = Path("/content/drive/MyDrive/Project Deeplearning/TACO")

from pathlib import Path

ANN_FILE = Path("/content/drive/MyDrive/Project Deeplearning/TACO/train_annotations.json")
# หรือจะใช้ SPLITS["train"] ภายหลังก็ได้

with open(ANN_FILE) as f:
    taco_json = json.load(f)

category_map = {cat['id']: cat['name'] for cat in taco_json['categories']}
print(f"✓ category_map loaded — {len(category_map)} classes")


✓ category_map loaded — 60 classes


In [5]:
from torch.utils.data import Dataset
class TacoDataset(Dataset):
    def __init__(self, root, annotations: dict, transforms=None):
        self.root       = root                      # Path/str ของโฟลเดอร์รูป
        self.transforms = transforms
        self.annos      = annotations["annotations"]
        self.images     = annotations["images"]
        self.id2img     = {img["id"]: img for img in self.images}

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = os.path.join(self.root, img_info["file_name"])
        image    = Image.open(img_path).convert("RGB")

        # ----- annotations ของภาพนี้ -----
        annos = [a for a in self.annos if a["image_id"] == img_info["id"]]

        boxes, labels = [], []
        for a in annos:
            x, y, w, h = a["bbox"]
            boxes.append([x, y, x + w, y + h])
            labels.append(a["category_id"])

        boxes  = torch.as_tensor(boxes,  dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes":    boxes,
            "labels":   labels,
            "image_id": torch.tensor([img_info["id"]]),   # ใช้ id จริง
            "area":     (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
            "iscrowd":  torch.zeros((len(boxes),), dtype=torch.int64),
        }

        # ---- transforms ----
        if self.transforms:
            image = self.transforms(image)
        # ถ้า transforms ไม่ได้แปลงเป็น tensor ให้ทำ
        if not isinstance(image, torch.Tensor):
            image = F.to_tensor(image)

        return image, target



In [6]:
import json                           # ← ถ้ายังไม่ใส่
from torchvision.transforms import functional as F
from torch.utils.data import DataLoader

ROOT = "/content/drive/MyDrive/Project Deeplearning/TACO"

# --- train ---
with open(f"{ROOT}/train_annotations.json") as f:
    train_json = json.load(f)

train_dataset = TacoDataset(
    root=f"{ROOT}/train",
    annotations=train_json,
    transforms=lambda img: F.to_tensor(img),
)

# --- val ---
with open(f"{ROOT}/val_annotations.json") as f:
    val_json = json.load(f)

val_dataset = TacoDataset(
    root=f"{ROOT}/val",
    annotations=val_json,
    transforms=lambda img: F.to_tensor(img),
)

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=2,
                          shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=1,
                          shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f"train: {len(train_dataset)} • val: {len(val_dataset)}")



train: 1050 • val: 225


In [7]:
import torch, torchvision
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_model(num_classes: int):
    # ── 1) load pretrained Faster-R-CNN-V2 ────────────────────────────
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1
    model   = fasterrcnn_resnet50_fpn_v2(weights=weights)

    # ── 2) replace the classifier head ────────────────────────────────
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)

    # ── 3) partial fine-tuning  (unfreeze layer4 ของ backbone) ────────
    for p in model.backbone.parameters():
        p.requires_grad_(False)

    for name, p in model.backbone.body.named_parameters():
        if "layer4" in name:
            p.requires_grad_(True)

    return model.to(device)

# -------- create model --------
num_classes = len(category_map) + 1     # +1 background
model = get_model(num_classes)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth
100%|██████████| 167M/167M [00:00<00:00, 198MB/s]


In [8]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ── เตรียม COCO ground-truth ของ val set (ทำครั้งเดียว) ──
coco_gt = COCO(f"{ROOT}/val_annotations.json")   # ← path val json

def evaluate_map(model, data_loader, device, coco_gt):
    model.eval()
    results = []

    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            for tgt, out in zip(targets, outputs):
                img_id = int(tgt["image_id"].item())

                boxes   = out["boxes"].cpu()
                scores  = out["scores"].cpu()
                labels  = out["labels"].cpu()

                # xyxy → xywh
                boxes[:, 2:] -= boxes[:, :2]

                for box, score, label in zip(boxes, scores, labels):
                    results.append({
                        "image_id"   : img_id,
                        "category_id": int(label.item()),
                        "bbox"       : [round(x, 2) for x in box.tolist()],
                        "score"      : float(score.item()),
                    })

    if not results:                         # กันกรณีโมเดลยังไม่พยากรณ์อะไร
        return 0.0
    # ── แพตช์ให้มีฟิลด์ info (ถ้าไม่มี) ──

    coco_gt.dataset.setdefault("info", {})
    coco_dt   = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval.evaluate(); coco_eval.accumulate(); coco_eval.summarize()

    return coco_eval.stats[0]               # mAP@0.5:0.95


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


In [10]:
params = [p for p in model.parameters() if p.requires_grad]   # <– ย้ำให้มีตัวแปรนี้
optimizer = optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=0.005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [ ]:
from torch.optim.lr_scheduler import StepLR
from tqdm import tqdm   # (ติดตั้ง tqdm ไปแล้ว)


num_epochs = 20
loss_hist, map_hist = [], []          # ← เก็บสถิติไว้ดูกราฟ

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, targets in tqdm(train_loader, desc=f"E{epoch+1}/{num_epochs}"):
        images  = [img.to(device) for img in images]
        targets = [{k:v.to(device) for k,v in t.items()} for t in targets]

        loss = sum(model(images, targets).values())
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)
        optimizer.step()
        running_loss += loss.item()

    lr_scheduler.step()
    avg_loss = running_loss / len(train_loader)
    print(f"[Epoch{epoch+1}] avg loss = {avg_loss:.4f}")

    map5095 = evaluate_map(model, val_loader, device, coco_gt)
    print(f"[Epoch{epoch+1}] mAP@[0.5:0.95] = {map5095:.4f}\n")


# ────────────────── วาดกราฟหลังฝึกจบ ──────────────────
epochs = range(1, num_epochs+1)

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(epochs, loss_hist, marker='o')
plt.title("Training Loss per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Avg Loss")

plt.subplot(1,2,2)
plt.plot(epochs, map_hist, marker='o')
plt.title("mAP@0.5:0.95 per Epoch")
plt.xlabel("Epoch")
plt.ylabel("mAP")

plt.tight_layout()
plt.show()



E1/20:  15%|█▌        | 79/525 [01:04<05:51,  1.27it/s]